<a href="https://colab.research.google.com/github/julaneelee/In-house-data-cleaning/blob/main/data_cleaning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#🌈1️⃣ **GE invitation email list **

### **1)**

## 🩷 from MRS - author

In [ ]:
from os import read
import pandas as pd
from google.colab import files

#Ask user to upload file
print("📂 Please upload your MRS-author excel file:")
uploaded = files.upload()

# Get the uploaded file name (first key in dict)
MRS_author_excel = list(uploaded.keys())[0]

# Step 2: Read only 'EM' column
MRS_author_df = pd.read_excel(MRS_author_excel)
#print(MRS_author_df.head())
print(MRS_author_df.columns.tolist())
print(len(MRS_author_df))

#MRS_author_excel=str(input("What is the path of MRS-author excel file in COLAB?: "))
# Load the entire Excel file
#print(MRS_author_excel)
#MRS_author_df = pd.read_excel(MRS_author_excel, sheet_name="Authors")
# print(MRS_author_df.head())


📂 Please upload your MRS-author excel file:


Saving 5.1.2026-9.1.2026-all.xlsx to 5.1.2026-9.1.2026-all (1).xlsx
['Name', 'Last Name', 'Title', 'Email', 'Is correspondence', 'Journal', 'Issue', 'Section', 'SI', 'Msid', 'Title.1', 'Doi', 'Website', 'Susy link', 'Published date', 'Affiliation', 'Country And Region', 'Keywords', 'AE Name', 'AE Email', 'Owner Name', 'Owner Email', 'Author Type', 'Author Type Reason']
89491


In [ ]:
from os import read
import pandas as pd
import re

# Remove rows where 'EMail' is NaN or blank
MRS_author_df = MRS_author_df[MRS_author_df['Email'].notna() & (MRS_author_df['Email'].str.strip() != '')]
# Remove duplicate rows based on the 'EMail' column
MRS_author_df = MRS_author_df.drop_duplicates(subset='Email', keep='first').reset_index(drop=True)

c_author_to_keep = ['Email', 'Country And Region']
df_author_column = MRS_author_df[c_author_to_keep]
#print(df_author_column.head())
print(len(df_author_column))

#*************************************************************
#I want only Zone1

df_author_column.loc[:, 'Country And Region'] = (
    df_author_column['Country And Region'].astype(str).str.strip().str.lower()
)
# List of countries to keep
countries_to_keep = [
    "Australia","Austria","Belgium","Bulgaria","Canada","Croatia","Cyprus",
    "Czech Republic","Denmark","Estonia","Finland","England","Scotland","France","Germany","Greece",
    "Hungary","Iceland","Ireland","Italy","Japan","Korea","South Korea","Republic of Korea","Latvia",
    "Liechtenstein","Lithuania","Luxembourg","Malta","Netherlands","the Netherlands","New Zealand",
    "Norway","Poland","Portugal","Romania","Slovakia","Slovenia",
    "Spain","Sweden","Switzerland","United Kingdom","United States","USA","Usa","UK","Uk",
    "Hong Kong","Singapore"
]

# Convert the list of countries to lowercase
countries_to_keep_lower = [c.lower() for c in countries_to_keep]

# Filter the dataframe ignoring case
df_author_filtered_zone1 = df_author_column[
    df_author_column['Country And Region'].isin(countries_to_keep_lower)].reset_index(drop=True)

#print(df_author_filtered_zone1.head())
#print(len(df_author_filtered_zone1))

#*********************************************************************

big_domains = [
    "gmail.com", "outlook.com", "yahoo.com", "icloud.com",
    "hotmail.com", "naver.com", "windowslive.com", "hanmail.com","googlemail.com","samsung.com",
    "live.com", "aol.com", "msn.com", "mail.com", "me.com", "mac.com"
]
df_author_filtered_zone1 = df_author_filtered_zone1[
    ~df_author_filtered_zone1['Email'].str.contains("student", flags=re.IGNORECASE, na=False)
].copy()

# ดึง domain หลัง @
df_author_filtered_zone1['Email_domain'] = df_author_filtered_zone1['Email'].str.split('@').str[-1].str.lower().str.strip()
#print(df_filtered_email['Email_domain'] )

# big domains + country-specific TLD
allowed_domains = big_domains + [
    'au','ac', 'at', 'be', 'bg', 'ca', 'hr', 'cy', 'cz', 'dk', 'ee', 'fi', 'fr', 'de', 'gr',
    'hu', 'is', 'ie', 'il', 'it', 'jp', 'kr', 'lv', 'li', 'lt', 'lu', 'mt', 'nl', 'nz',
    'no', 'pl', 'pt', 'ro', 'sk', 'si', 'es', 'se', 'ch', 'uk', 'us', 'hk', 'sg','eu','edu','org','net'
]
allowed_domains_pattern = "|".join([re.escape(d) for d in allowed_domains])

df_author_filtered_zone1_default  = df_author_filtered_zone1[
    df_author_filtered_zone1['Email_domain'].str.contains(
        f"(?:{allowed_domains_pattern})$", flags=re.IGNORECASE, na=False
    )
].reset_index(drop=True)

df_author_filtered_zone1_default = df_author_filtered_zone1_default.drop(columns=['Email_domain'])
#print(df_author_filtered_zone1_default.head())
print(len(df_author_filtered_zone1_default))


71995
55150


In [ ]:
print(type(df_author_filtered_zone1_default))

<class 'pandas.core.frame.DataFrame'>


In [ ]:
import pandas as pd
import math
import os

# ===============================
# CONFIG
# ===============================
MAX_ROWS = 18000
OUTPUT_PREFIX = "author_zone1-emails"   # ชื่อไฟล์ที่อยากได้

# ===============================
# DATAFRAME
# ===============================
df = df_author_filtered_zone1_default.copy()

# ===============================
# KEEP ONLY EMAIL COLUMN
# ===============================
df = df[["email"]]   # ลบ country, region และ column อื่น ๆ อัตโนมัติ

# ===============================
# SPLIT & EXPORT TO EXCEL
# ===============================
total_rows = len(df)

if total_rows <= MAX_ROWS:
    df.to_excel(f"{OUTPUT_PREFIX}_1.xlsx", index=False)
    print(f"Saved 1 file with {total_rows} rows")

else:
    num_files = math.ceil(total_rows / MAX_ROWS)

    for i in range(num_files):
        start_row = i * MAX_ROWS
        end_row = start_row + MAX_ROWS

        df_chunk = df.iloc[start_row:end_row]
        df_chunk.to_excel(f"{OUTPUT_PREFIX}_{i + 1}.xlsx", index=False)

    print(f"Saved {num_files} files (each ≤ {MAX_ROWS} rows)")

# ===============================
# SHOW SAVE LOCATION
# ===============================
print("Files saved at:", os.getcwd())


Saved 4 files (each ≤ 18000 rows)
Files saved at: /content


In [ ]:

# Export file - only Author emails

Author_email_only = df_author_filtered_zone1_default["Email"].astype(str).str.encode('ascii', 'ignore').str.decode('ascii').str.strip().to_frame()
#Author_email_only = pd.DataFrame(Author_email_only, columns=["Email"])
#print(Author_email_only)

# Ask user for file name
Author_email_only_filename = input("Enter the name for the Excel file (without extension): ").strip()

# Ensure the file name ends with .xlsx
if not Author_email_only_filename.endswith(".xlsx"):
    Author_email_only_filename = Author_email_only_filename + ".xlsx"

# Export file
Author_email_only.to_excel(Author_email_only_filename, index=False, engine="openpyxl")
print(f"File saved as: {Author_email_only_filename}")
print(f"Exported {len(Author_email_only)} rows to {Author_email_only_filename}")


KeyboardInterrupt: Interrupted by user

## 🩷 from Scilit

In [ ]:
from os import read
import pandas as pd
from google.colab import files

#Ask user to upload file
print("📂 Please upload your Scilit excel file :")
uploaded_scilit = files.upload()

# Get the uploaded file name (first key in dict)
Scilit_excel = list(uploaded_scilit.keys())[0]

Scilit_excel_df = pd.read_excel(Scilit_excel)
#print(Scilit_excel_df.head())
print(Scilit_excel_df.columns.tolist())
print(len(Scilit_excel_df))

📂 Please upload your Scilit excel file :


Saving 19.11-scilit.xlsx to 19.11-scilit.xlsx
['Name', 'Affiliation', 'Country/Region', 'Email', 'Author Link', 'H Index', 'Article', 'Article Link', 'Total Citations', 'Published year', 'Journal title', 'Volume', 'First Page', 'Last Page', 'Keywords', 'Abstract']
57424


In [ ]:
from os import read
import pandas as pd
import re

# Remove rows where 'EMail' is NaN or blank
Scilit_excel_df = Scilit_excel_df[Scilit_excel_df['Email'].notna() & (Scilit_excel_df['Email'].str.strip() != '')]
# Remove duplicate rows based on the 'EMail' column
Scilit_excel_df = Scilit_excel_df.drop_duplicates(subset='Email', keep='first').reset_index(drop=True)

print(Scilit_excel_df.columns.tolist())

['Name', 'Affiliation', 'Country/Region', 'Email', 'Author Link', 'H Index', 'Article', 'Article Link', 'Total Citations', 'Published year', 'Journal title', 'Volume', 'First Page', 'Last Page', 'Keywords', 'Abstract']


In [ ]:
cs_author_to_keep = ['Email','Country/Region']
df_scilit_column = Scilit_excel_df[cs_author_to_keep]
#print(df_scilit_column)
#print(Scilit_excel_df.head())

#*************************************************************
#I want only Zone1

df_scilit_column.loc[:, 'Country/Region'] = (
    df_scilit_column['Country/Region'].astype(str).str.strip().str.lower()
)
# List of countries to keep
countries_to_keep = [
    "Australia","Austria","Belgium","Bulgaria","Canada","Croatia","Cyprus",
    "Czech Republic","Denmark","Estonia","Finland","England","Scotland","France","Germany","Greece",
    "Hungary","Iceland","Ireland","Italy","Japan","Korea","South Korea","Republic of Korea","Latvia",
    "Liechtenstein","Lithuania","Luxembourg","Malta","Netherlands","the Netherlands","New Zealand",
    "Norway","Poland","Portugal","Romania","Slovakia","Slovenia",
    "Spain","Sweden","Switzerland","United Kingdom","United States","USA","Usa","UK","Uk",
    "Hong Kong","Singapore"
]

# Convert the list of countries to lowercase
countries_to_keep_lower = [c.lower() for c in countries_to_keep]

# Filter the dataframe ignoring case
Scilit_excel_df = df_scilit_column[
    df_scilit_column['Country/Region'].isin(countries_to_keep_lower)].reset_index(drop=True)

#print(Scilit_excel_df.head())
#print(len(Scilit_excel_df))

#*********************************************************************

big_domains = [
    "gmail.com", "outlook.com", "yahoo.com", "icloud.com",
    "hotmail.com", "naver.com", "windowslive.com", "hanmail.com","googlemail.com","samsung.com",
    "live.com", "aol.com", "msn.com", "mail.com", "me.com", "mac.com"
]
Scilit_excel_df = Scilit_excel_df[
    ~Scilit_excel_df['Email'].str.contains("student", flags=re.IGNORECASE, na=False)
].copy()

# ดึง domain หลัง @
Scilit_excel_df['Email_domain'] = Scilit_excel_df['Email'].str.split('@').str[-1].str.lower().str.strip()
#print(df_filtered_email['Email_domain'] )

# big domains + country-specific TLD
allowed_domains = big_domains + [
    'au','ac', 'at', 'be', 'bg', 'ca', 'hr', 'cy', 'cz', 'dk', 'ee', 'fi', 'fr', 'de', 'gr',
    'hu', 'is', 'ie', 'il', 'it', 'jp', 'kr', 'lv', 'li', 'lt', 'lu', 'mt', 'nl', 'nz',
    'no', 'pl', 'pt', 'ro', 'sk', 'si', 'es', 'se', 'ch',  'uk', 'us', 'hk', 'sg','eu','edu','org','net'
]
allowed_domains_pattern = "|".join([re.escape(d) for d in allowed_domains])

Scilit_excel_df_default  = Scilit_excel_df[
    Scilit_excel_df['Email_domain'].str.contains(
        f"(?:{allowed_domains_pattern})$", flags=re.IGNORECASE, na=False
    )
].reset_index(drop=True)

Scilit_excel_df_default = Scilit_excel_df_default.drop(columns=['Email_domain'])
print(Scilit_excel_df_default.head())
print(len(Scilit_excel_df_default))

NameError: name 'Scilit_excel_df' is not defined

In [ ]:
# Export file - only Author emails

scilit_email_only = Scilit_excel_df_default["Email"].astype(str).str.encode('ascii', 'ignore').str.decode('ascii').str.strip().to_frame()
#scilit_email_only = pd.DataFrame(scilit_email_only, columns=["Email"])
#print(scilit_email_only)


# Ask user for file name
scilit_email_only_filename = input("Enter the name for the Excel file (without extension): ").strip()
# Ensure the file name ends with .xlsx
if not scilit_email_only_filename.endswith(".xlsx"):
   scilit_email_only_filename = scilit_email_only_filename + ".xlsx"

scilit_email_only.to_excel(scilit_email_only_filename, index=False)
# Step 7: Download the file
files.download(scilit_email_only_filename)

print(f"File saved as: {scilit_email_only_filename}")
print(f"Exported {len(scilit_email_only)} rows to {scilit_email_only_filename}")

Enter the name for the Excel file (without extension): 19.11-scilit-mail


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File saved as: 19.11-scilit-mail.xlsx
Exported 21725 rows to 19.11-scilit-mail.xlsx


In [ ]:
# Export file - only Author emails

##scilit_email_only = Scilit_excel_df_default["Email"].astype(str).str.encode('ascii', 'ignore').str.decode('ascii').str.strip().to_frame()

# Ask user for file name
##scilit_email_only_filename = input("Enter the name for the Excel file (without extension): ").strip()

# Ensure the file name ends with .xlsx
##if not scilit_email_only_filename.endswith(".xlsx"):
##   scilit_email_only_filename = scilit_email_only_filename + ".xlsx"

# Export file
##scilit_email_only.to_excel(scilit_email_only_filename, index=False, engine="openpyxl")
##print(f"File saved as: {scilit_email_only_filename}")
##print(f"Exported {len(scilit_email_only)} rows to {scilit_email_only_filename}")


Enter the name for the Excel file (without extension): Scilit-3.9-GE-invitation-list
File saved as: Scilit-3.9-GE-invitation-list.xlsx
Exported 25815 rows to Scilit-3.9-GE-invitation-list.xlsx


## 🩷from Web of Science

In [ ]:
import pandas as pd
from google.colab import files

# Step 1: Ask user to upload file
print("📂 Please upload your Excel file:")
uploaded = files.upload()

# Get the uploaded file name (first key in dict)
file_name = list(uploaded.keys())[0]

# Step 2: Read only 'EM' column
df = pd.read_excel(file_name, usecols=["EM"])

# Step 3: Split emails by semicolon and explode
df_clean = df['EM'].str.split(r'\s*;\s*').explode().reset_index(drop=True)

# Step 4: Put into DataFrame
df_clean = pd.DataFrame(df_clean, columns=["EM"])


# Step 5: Remove blanks/NaN + duplicates
df_clean = df_clean[df_clean['EM'].notna() & (df_clean['EM'].str.strip() != "")]
df_clean = df_clean.drop_duplicates().reset_index(drop=True)


# Step 6: Save cleaned file
output_file = "emails_cleaned.xlsx"
df_clean.to_excel(output_file, index=False)

# Step 7: Download the file
files.download(output_file)


📂 Please upload your Excel file:


Saving 18.9-WOS-crude&1.xlsx to 18.9-WOS-crude&1.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **2)**

## 💛After we get the file for export and upload to MRS-Editor-invited/SCILIT to get H-index, after get the file from MRS then REUPLOAD/IMPORT FILE TO HERE AGAIN!!!!

In [ ]:
from os import read
import pandas as pd
from google.colab import files

#Ask user to upload file
print("📂 Please upload your excel file :")
uploaded_MRS_editor_excel = files.upload()

# Get the uploaded file name (first key in dict)
MRS_editor_excel = list(uploaded_MRS_editor_excel.keys())[0]

MRS_editor_df = pd.read_excel(MRS_editor_excel)
#print(MRS_editor_df.head())
print(MRS_editor_df.columns.tolist())
print(len(MRS_editor_df))

📂 Please upload your excel file :


Saving merged_file.xlsx to merged_file (3).xlsx
['Email', 'Linked Email', 'Name', 'Affiliation', 'Homepage', 'H-index', 'H-index link', 'Invited Date', 'Status', 'Invited Numbers', 'Block from editing Special Issue/Topic Collection as Guest Editor', 'Block from receiving CfPs and similar emails', 'Block from reviewing manuscripts', 'Automated blacklist for CFP', 'Block from newsletters and cooperate informations', 'Block from receiving participation emails from Sciforum', 'Block from being Topical Advisory Panel/Editorial Board Member', 'Block from receiving employee recruit emails', 'Receive MDPI Books Mailing']
55150


In [ ]:

# Remove rows where 'EMail' is NaN or blank
MRS_editor_df = MRS_editor_df[MRS_editor_df['Email'].notna() & (MRS_editor_df['Email'].str.strip() != '')]
# Remove duplicate rows based on the 'EMail' column
MRS_editor_df = MRS_editor_df.drop_duplicates(subset='Email', keep='first').reset_index(drop=True)

# Columns to check
cols_to_check_block = ['Block from editing Special Issue/Topic Collection as Guest Editor',
                       'Block from receiving CfPs and similar emails',
                       'Block from reviewing manuscripts',
                       'Automated blacklist for CFP',
                       'Block from newsletters and cooperate informations',
                       'Block from receiving participation emails from Sciforum',
                       'Block from being Topical Advisory Panel/Editorial Board Member',
                       'Block from receiving employee recruit emails',
                       'Receive MDPI Books Mailing'
]

# Keep only rows where none of the columns contain "Yes"
MRS_editor_df_filtered = MRS_editor_df[~MRS_editor_df[cols_to_check_block].apply(lambda row: row.str.contains("Yes", case=False, na=False)).any(axis=1)]
MRS_editor_df_filtered = MRS_editor_df_filtered.drop(columns= cols_to_check_block)
#print(MRS_editor_df_filtered.head())
status_GE = ["Guest Editor - Invited",
             "Guest Editor - Uninvited",
             "Guest Editor - Rejected",
             "Topic Editor - Invited",
             "Section Board Member - Invited",
             "Section Board Member - Uninvited",
             "Section Board Member - Rejected",
             "Editorial Board Member - Invited",
             "Editorial Board Member - Uninvited",
             "Editorial Board Member - Rejected",
             "RB-Invited",
             "RB-Uninvited",
             "RB-Rejected",
             "Topical Advisory Panel Member - Invited",
             "Topical Advisory Panel Member - Uninvited",
             "Topical Advisory Panel Member - Rejected",""," "
]

# Keep only rows where column Status is in the list
# Keep only rows where column Status is in the list OR blank
MRS_editor_df_filtered = MRS_editor_df_filtered[
    MRS_editor_df_filtered['Status'].isin(status_GE) |
    (MRS_editor_df_filtered['Status'].isna()) |
    (MRS_editor_df_filtered['Status'].str.strip() == '')
]

#print(MRS_editor_df_filtered.columns.to_list())
MRS_editor_df_filtered = MRS_editor_df_filtered[['Email', 'Name', 'H-index', 'Invited Date', 'Status', 'Invited Numbers']]
print(MRS_editor_df_filtered.head())
print(len(MRS_editor_df_filtered))


                        Email              Name H-index Invited Date Status  \
0     cojocaruion90@yahoo.com               NaN     NaN          NaN    NaN   
1  guliciuc.madalin@gmail.com  Mădălin Guliciuc       1          NaN    NaN   
2    salinschielena@gmail.com               NaN     NaN          NaN    NaN   
3     cristina.serban@ugal.ro   Cristina Şerban       5          NaN    NaN   
4   pascaru.grigore@yahoo.com   Grigore Pascaru     NaN          NaN    NaN   

   Invited Numbers  
0                0  
1                0  
2                0  
3                0  
4                0  
45059


In [ ]:
print(MRS_editor_df_filtered.head())
print(MRS_editor_df_filtered.columns)

                        Email              Name H-index Invited Date Status  \
0     cojocaruion90@yahoo.com               NaN     NaN          NaN    NaN   
1  guliciuc.madalin@gmail.com  Mădălin Guliciuc       1          NaN    NaN   
2    salinschielena@gmail.com               NaN     NaN          NaN    NaN   
3     cristina.serban@ugal.ro   Cristina Şerban       5          NaN    NaN   
4   pascaru.grigore@yahoo.com   Grigore Pascaru     NaN          NaN    NaN   

   Invited Numbers  
0                0  
1                0  
2                0  
3                0  
4                0  
Index(['Email', 'Name', 'H-index', 'Invited Date', 'Status',
       'Invited Numbers'],
      dtype='object')


In [ ]:
import pandas as pd

# Ensure Invited Date is datetime type
MRS_editor_df_filtered['Invited Date'] = pd.to_datetime(MRS_editor_df_filtered['Invited Date'], errors='coerce')

# 1. Create new column with +90 days and +5 hours
MRS_editor_df_filtered['Available Date (+90d6hrs)'] = (
    MRS_editor_df_filtered['Invited Date'] + pd.Timedelta(days=90, hours=6)
)

# 2. Ask user for H-index range
min_h = float(input("Enter minimum H-index to keep: "))
max_h = float(input("Enter maximum H-index to keep: "))

# Convert H-index to numeric, remove non-numeric or blank
MRS_editor_df_filtered['H-index'] = pd.to_numeric(MRS_editor_df_filtered['H-index'], errors='coerce')

# Filter H-index dynamically
MRS_editor_df_filtered = MRS_editor_df_filtered[
    (MRS_editor_df_filtered['H-index'] >= min_h) &
    (MRS_editor_df_filtered['H-index'] <= max_h)
]

# 3. Ask user for max Invited Numbers
max_invited = float(input("Enter maximum Invited Numbers to keep (blank counted as 0): "))

# Convert Invited Numbers to numeric, fill blank/NaN with 0
MRS_editor_df_filtered['Invited Numbers'] = pd.to_numeric(MRS_editor_df_filtered['Invited Numbers'], errors='coerce').fillna(0)

# Filter Invited Numbers dynamically
MRS_editor_df_filtered = MRS_editor_df_filtered[MRS_editor_df_filtered['Invited Numbers'] <= max_invited]



# Show result
print(MRS_editor_df_filtered.head())
print(len(MRS_editor_df_filtered))

Enter minimum H-index to keep: 9
Enter maximum H-index to keep: 40
Enter maximum Invited Numbers to keep (blank counted as 0): 10
                    Email                  Name  H-index        Invited Date  \
18  whwjdzl1004@gmail.com          Junghwan Cho     14.0 2025-11-12 03:21:30   
41   birbilis@med.duth.gr  Theodossios Birbilis     11.0 2025-11-03 07:54:14   
49    kean.hsu@nus.edu.sg           Kean J. Hsu     13.0 2025-11-27 04:47:44   
50     nadyanna@u.nus.edu       Nadyanna Majeed     14.0                 NaT   
54   mislavorto@gmail.com          Mislav Jelić     19.0 2025-11-11 10:54:23   

                      Status  Invited Numbers Available Date (+90d6hrs)  
18   Guest Editor - Rejected                1       2026-02-10 09:21:30  
41    Guest Editor - Invited                7       2026-02-01 13:54:14  
49    Guest Editor - Invited                7       2026-02-25 10:47:44  
50                       NaN                0                       NaT  
54  Guest Editor - 

In [ ]:
#Ask user for cutoff date (DD.MM.YYYY) and filter Invited Date Plus 90d5h
user_date_input = input("Enter cutoff date (DD.MM.YYYY): ")
cutoff_date = pd.to_datetime(user_date_input, format="%d.%m.%Y")

# Extend cutoff to end of day (23:59:59.999999)
cutoff_date = cutoff_date + pd.Timedelta(days=1) - pd.Timedelta(microseconds=1)

# Keep only rows where Invited Date Plus 90d5h is before or equal to cutoff
new_filtered_df_invite = MRS_editor_df_filtered[
    (MRS_editor_df_filtered['Available Date (+90d6hrs)'] <= cutoff_date) |
    (MRS_editor_df_filtered['Available Date (+90d6hrs)'].isna())
].copy()

#new_filtered_df_invite = new_filtered_df_invite.drop(columns=['Invited Data'])

# Display result
print(new_filtered_df_invite)
print("Total rows after all filtering:", len(new_filtered_df_invite))

Enter cutoff date (DD.MM.YYYY): 6.1.2026
                            Email                  Name  H-index  \
50             nadyanna@u.nus.edu       Nadyanna Majeed     14.0   
214           yujin.kim@snu.ac.kr             Yujin Kim     39.0   
369            vmehrnoush@nosm.ca       Vahid Mehrnoush     10.0   
374        omar.abdelhadi@tbh.net        Omar AbdelAziz     23.0   
432           anh.dao.hoang@ut.ee           D. A. Hoang      9.0   
...                           ...                   ...      ...   
54780      crisgr_xxi@hotmail.com  Cristina García Ruiz     12.0   
54911  andreas.voetsch@icloud.com        Vötsch Andreas     11.0   
54974            soriano@uchc.edu          Jose Soriano     13.0   
55033             sgd49@miami.edu          Scott Daniel     12.0   
55135   sezai.ozkan@radboudumc.nl           Özkan Sezai     16.0   

             Invited Date                    Status  Invited Numbers  \
50                    NaT                       NaN                0  

In [ ]:

# Ask user for file name
GE_invite_filename = input("Enter the name for GE invitation file (without extension): ").strip()
# Ensure the file name ends with .xlsx
if not GE_invite_filename.endswith(".xlsx"):
    GE_invite_filename = GE_invite_filename + ".xlsx"

new_filtered_df_invite.to_excel(GE_invite_filename, index=False)
# Step 7: Download the file
files.download(GE_invite_filename)

print(f"File saved as: {GE_invite_filename}")
print(f"Exported {len(new_filtered_df_invite)} rows to {GE_invite_filename}")

Enter the name for GE invitation file (without extension): 6.1.2026-GE-inv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File saved as: 6.1.2026-GE-inv.xlsx
Exported 645 rows to 6.1.2026-GE-inv.xlsx


# *แบ่งไฟล์ใหญ่เป็นไฟล์ย่อย - กรณีมีแค่ Email column เดี่ยวๆ ในไฟล์แรก*

In [ ]:
import pandas as pd
import math
import os
from google.colab import files

# ===============================
# CONFIG
# ===============================
MAX_ROWS = 18000
OUTPUT_PREFIX = "author_email_only"

# ===============================
# STEP 1: UPLOAD EXCEL FILE
# ===============================
print("📂 Please upload your author Excel file:")
uploaded = files.upload()

# Get uploaded file name
excel_file = list(uploaded.keys())[0]

# ===============================
# STEP 2: READ EXCEL INTO DATAFRAME
# ===============================
df = pd.read_excel(excel_file)

print(f"Loaded file: {excel_file}")
print(f"Total rows before cleaning: {len(df)}")

# ===============================
# STEP 3: KEEP ONLY EMAIL COLUMN
# ===============================
df = df[["Email"]]

# Optional: remove empty or duplicated emails
df = df.dropna(subset=["Email"])
df = df.drop_duplicates(subset=["Email"])

print(f"Total rows after cleaning: {len(df)}")

# ===============================
# STEP 4: SPLIT & EXPORT TO EXCEL
# ===============================
total_rows = len(df)

if total_rows <= MAX_ROWS:
    df.to_excel(f"{OUTPUT_PREFIX}_1.xlsx", index=False)
    print(f"Saved 1 file with {total_rows} rows")

else:
    num_files = math.ceil(total_rows / MAX_ROWS)

    for i in range(num_files):
        start_row = i * MAX_ROWS
        end_row = start_row + MAX_ROWS

        df_chunk = df.iloc[start_row:end_row]
        df_chunk.to_excel(f"{OUTPUT_PREFIX}_{i + 1}.xlsx", index=False)

    print(f"Saved {num_files} files (each ≤ {MAX_ROWS} rows)")

# ===============================
# STEP 5: SHOW SAVE LOCATION
# ===============================
print("Files saved at:", os.getcwd())


📂 Please upload your author Excel file:


Saving 5.1.2026-9.1.2026-email.xlsx to 5.1.2026-9.1.2026-email (4).xlsx
Loaded file: 5.1.2026-9.1.2026-email (4).xlsx
Total rows before cleaning: 55150
Total rows after cleaning: 55150
Saved 4 files (each ≤ 18000 rows)
Files saved at: /content


# แบ่งไฟล์ใหญ่ให้เป็นไฟล์ย่อย

In [ ]:
import pandas as pd
import math
import os
from google.colab import files

# ===============================
# CONFIG
# ===============================
MAX_ROWS = 18000
OUTPUT_PREFIX = "split_file"

# ===============================
# STEP 1: UPLOAD EXCEL FILE
# ===============================
print("📂 Please upload your Excel file:")
uploaded = files.upload()

excel_file = list(uploaded.keys())[0]

# ===============================
# STEP 2: READ EXCEL (ALL COLUMNS)
# ===============================
df = pd.read_excel(excel_file)

print(f"Loaded file: {excel_file}")
print(f"Total rows: {len(df)}")

# ===============================
# STEP 3: SPLIT & EXPORT BY ROWS
# ===============================
total_rows = len(df)

if total_rows <= MAX_ROWS:
    df.to_excel(f"{OUTPUT_PREFIX}_1.xlsx", index=False)
    print(f"Saved 1 file with {total_rows} rows")

else:
    num_files = math.ceil(total_rows / MAX_ROWS)

    for i in range(num_files):
        start_row = i * MAX_ROWS
        end_row = start_row + MAX_ROWS

        df_chunk = df.iloc[start_row:end_row]
        df_chunk.to_excel(f"{OUTPUT_PREFIX}_{i + 1}.xlsx", index=False)

    print(f"Saved {num_files} files (each ≤ {MAX_ROWS} rows)")

# ===============================
# STEP 4: SHOW SAVE LOCATION
# ===============================
print("Files saved at:", os.getcwd())


# รวมไฟล์ย่อยเป็นไฟล์ใหญ่

In [ ]:
import pandas as pd
import os
from google.colab import files

# ===============================
# CONFIG
# ===============================
OUTPUT_FILE = "merged_file.xlsx"

# ===============================
# STEP 1: UPLOAD MULTIPLE FILES
# ===============================
print("📂 Please upload Excel files to merge (you can select multiple files):")
uploaded = files.upload()

# ===============================
# STEP 2: READ & COLLECT DATAFRAMES
# ===============================
dfs = []
expected_columns = None

for file_name in uploaded.keys():
    df = pd.read_excel(file_name)

    # Check header consistency
    if expected_columns is None:
        expected_columns = list(df.columns)
    else:
        if list(df.columns) != expected_columns:
            raise ValueError(
                f"❌ Column mismatch in file: {file_name}\n"
                f"Expected: {expected_columns}\n"
                f"Found: {list(df.columns)}"
            )

    dfs.append(df)
    print(f"Loaded {file_name} ({len(df)} rows)")

# ===============================
# STEP 3: CONCATENATE ALL FILES
# ===============================
merged_df = pd.concat(dfs, ignore_index=True)

print(f"✅ Total merged rows: {len(merged_df)}")

# ===============================
# STEP 4: EXPORT TO EXCEL
# ===============================
merged_df.to_excel(OUTPUT_FILE, index=False)

print(f"📁 Merged file saved as: {OUTPUT_FILE}")
print("Files saved at:", os.getcwd())


📂 Please upload Excel files to merge (you can select multiple files):


Saving author_email_only_4-mrs.xlsx to author_email_only_4-mrs (1).xlsx
Saving author_email_only_3-mrs.xlsx to author_email_only_3-mrs (1).xlsx
Saving author_email_only_2-mrs.xlsx to author_email_only_2-mrs (1).xlsx
Saving author_email_only_1-mrs.xlsx to author_email_only_1-mrs (1).xlsx
Loaded author_email_only_4-mrs (1).xlsx (1150 rows)
Loaded author_email_only_3-mrs (1).xlsx (18000 rows)
Loaded author_email_only_2-mrs (1).xlsx (18000 rows)
Loaded author_email_only_1-mrs (1).xlsx (18000 rows)
✅ Total merged rows: 55150
📁 Merged file saved as: merged_file.xlsx
Files saved at: /content


--------------------------------------

#🌈2️⃣ **CFP- Email Invitation Preparation** from Purged - MDPI backend

## 1) Filter Email and country

Import file we get from MDPI backend (Scilit request) or call "Purged email" >> Manually merge 'notMatch' sheet to 'purged' sheet (remove grey highlight, keep green, blue, yellow) >> file contains only one sheet name 'purged' >> Save as new file >> UPLOAD TO COLAB >> Fill in the PATH IN CODE >> RUN

In [ ]:
# @title
from os import read
import pandas as pd
from google.colab import files

#Ask user to upload file
print("📂 Please upload your Crude file (export from purged backend):")
uploaded_file_crude = files.upload()

# Get the uploaded file name (first key in dict)
crude_excel = list(uploaded_file_crude.keys())[0]

df = pd.read_excel(crude_excel)
#print(df.head())
print(df.columns.tolist())
print(len(df))

📂 Please upload your Crude file (export from purged backend):


Saving purged.xlsx to purged.xlsx
['Name', 'EMail', 'Reference', 'Link', 'Google Scholar', 'TC', 'PY', 'Affiliation', 'Check', 'Country/Region', 'Publisher', 'Voucher amount', 'Central pay', 'Discount percent', 'H-Index']
19313


In [ ]:
columns_to_keep = ['Name', 'EMail', 'Reference', 'Affiliation', 'Country/Region']
df_filtered_column = df[columns_to_keep]

# Remove rows where 'EMail' is NaN or blank
df_filtered_column = df_filtered_column[df_filtered_column['EMail'].notna() & (df_filtered_column['EMail'].str.strip() != '')]

# Remove duplicate rows based on the 'EMail' column
df_filtered_column = df_filtered_column.drop_duplicates(subset='EMail', keep='first').reset_index(drop=True)

# df_filtered_column now has only unique, non-empty emails
# Display the result
print(df_filtered_column.head())
print(len(df_filtered_column))

        Name                                       EMail  \
0   Varghese                           bipintv@gmail.com   
1     Ozkent                          msozkent@gmail.com   
2      Atıcı                      ahmetatici06@gmail.com   
3  Lazarides  alexanderleandros.lazarides@sinaihealth.ca   
4       Chen                          acbbaz@sjtu.edu.cn   

                                           Reference  \
0  Head and neck surgical oncology training in th...   
1  The Benign Renal Masses that Were Exposed afte...   
2  The Benign Renal Masses that Were Exposed afte...   
3  How Do the Outcomes of Radiation-Associated Pe...   
4  Oncological Analysis and Surgical Outcomes in ...   

                                         Affiliation Country/Region  
0  Bipin T. Varghese (author), Department of Surg...          India  
1  Serkan Ozkent (author), Konya City Hosp, Dept ...         Turkey  
2  Ahmet Atıcı (author), Department of Urology, M...         Turkey  
3  Alexander L. Lazari

In [ ]:
#I want only Zone1

df_filtered_column.loc[:, 'Country/Region'] = (
    df_filtered_column['Country/Region'].astype(str).str.strip().str.lower()
)

# List of countries to keep
countries_to_keep = [
    "Australia","Austria","Belgium","Bulgaria","Canada","Croatia","Cyprus",
    "Czech Republic","Denmark","Estonia","Finland","England","Scotland","France","Germany","Greece",
    "Hungary","Iceland","Ireland","Italy","Japan","Korea","South Korea","Republic of Korea","Latvia",
    "Liechtenstein","Lithuania","Luxembourg","Malta","Netherlands","the Netherlands","New Zealand",
    "Norway","Poland","Portugal","Romania","Slovakia","Slovenia",
    "Spain","Sweden","Switzerland","United Kingdom","United States","USA","Usa","UK","Uk",
    "Hong Kong","Singapore"
]

# Convert the list of countries to lowercase
countries_to_keep_lower = [c.lower() for c in countries_to_keep]

# Filter the dataframe ignoring case
df_filtered_zone1 = df_filtered_column[
    df_filtered_column['Country/Region'].isin(countries_to_keep_lower)].reset_index(drop=True)

print(df_filtered_zone1.head())
print(len(df_filtered_zone1))

         Name                                         EMail  \
0   Lazarides    alexanderleandros.lazarides@sinaihealth.ca   
1     Murando  federica.murando@guest.policlinicogemelli.it   
2      Wilder                             c941w083@kumc.edu   
3  Al Zadjali                           drzadjali@gmail.com   
4     Magaldi                 michelangelomagaldi@gmail.com   

                                           Reference  \
0  How Do the Outcomes of Radiation-Associated Pe...   
1  Oncoplastic Breast Surgery versus Conservative...   
2  Improving Medical Student Anatomy Knowledge an...   
3  Surgical approaches for pleomorphic adenoma of...   
4  Minimal Margin Surgery and Intraoperative Neur...   

                                         Affiliation Country/Region  
0  Alexander L. Lazarides (author), Division of O...         canada  
1  Federica Murando (author), Fondazione Policlin...          italy  
2  Chloe Wilder (author), School of Medicine, Uni...            usa  
3  F

In [ ]:
# I want to remove company affiliation, and thoese non-related affiliation to clinical medicine

import re
# ลบคำเกี่ยวกับ company + animal + agriculture
pattern = r'\b(?:inc|ltd|animal|veterinary|agricult|agriculture|agricultural|forensic|chemistry|mechanical|electrical|food|foods)\b'

df_filtered_affiliation = df_filtered_zone1[
    ~df_filtered_zone1['Affiliation'].str.contains(pattern, flags=re.IGNORECASE, na=False)
].reset_index(drop=True)

print(df_filtered_affiliation)

             Name                                         EMail  \
0       Lazarides    alexanderleandros.lazarides@sinaihealth.ca   
1         Murando  federica.murando@guest.policlinicogemelli.it   
2          Wilder                             c941w083@kumc.edu   
3      Al Zadjali                           drzadjali@gmail.com   
4         Magaldi                 michelangelomagaldi@gmail.com   
...           ...                                           ...   
10474        Tran                       atran31@bwh.harvard.edu   
10475        Weng                        wenjinwe@med.umich.edu   
10476     Konishi                      adlibitum.kt63@gmail.com   
10477        Khan                       aymankhanmail@gmail.com   
10478      Kumata                      tohoku_a4mb1@yahoo.co.jp   

                                               Reference  \
0      How Do the Outcomes of Radiation-Associated Pe...   
1      Oncoplastic Breast Surgery versus Conservative...   
2      Improvin

In [ ]:
import re

big_domains = [
    "gmail.com", "outlook.com", "yahoo.com", "icloud.com",
    "hotmail.com", "naver.com", "windowslive.com", "hanmail.com","googlemail.com","samsung.com",
    "live.com", "aol.com", "msn.com", "mail.com", "me.com", "mac.com"
]

df_filtered_email = df_filtered_affiliation[
    ~df_filtered_affiliation['EMail'].str.contains("student", flags=re.IGNORECASE, na=False)
].copy()

# ดึง domain หลัง @
df_filtered_email['Email_domain'] = df_filtered_email['EMail'].str.split('@').str[-1].str.lower().str.strip()
#print(df_filtered_email['Email_domain'] )

# big domains + country-specific TLD
allowed_domains = big_domains + [
    'au','ac', 'at', 'be', 'bg', 'ca', 'hr', 'cy', 'cz', 'dk', 'ee', 'fi', 'fr', 'de', 'gr',
    'hu', 'is', 'ie', 'il', 'it', 'jp', 'kr', 'lv', 'li', 'lt', 'lu', 'mt', 'nl', 'nz',
    'no', 'pl', 'pt', 'ro', 'sk', 'si', 'es', 'se', 'ch', 'uk', 'us', 'hk', 'sg','eu','edu','org','net'
]
allowed_domains_pattern = "|".join([re.escape(d) for d in allowed_domains])

df_default  = df_filtered_email[
    df_filtered_email['Email_domain'].str.contains(
        f"(?:{allowed_domains_pattern})$", flags=re.IGNORECASE, na=False
    )
].reset_index(drop=True)

df_default = df_default.drop(columns=['Email_domain'])


In [ ]:
print(df_default.head())
print(len(df_default ))

       Name                      EMail  \
0     Hafez  hafez@vetmed.fu-berlin.de   
1  Cassetti   viola.cassetti@gmail.com   
2    Bowman       larry.bowman@uga.edu   
3   Catallo        ccatallo@ryerson.ca   
4   Staines  heather.staines@gmail.com   

                                           Reference  \
0  Public Health Impact of Staphylococcal and Clo...   
1  Facilitators and barriers to implement communi...   
2  Innovative Teaching for Resilience and Teamwor...   
3  A new approach to digital health? Virtual COVI...   
4  Searching for evidence in public health emerge...   

                                         Affiliation Country/Region  
0  Hafez Mohamed Hafez (author), Former Head of t...        germany  
1  Viola Cassetti (author), Valencian Internation...          spain  
2  Larry L. Bowman (author), East Tennessee State...            usa  
3  Cristina Catallo (author), Faculty of Communit...         canada  
4  Heather Staines (author), Delta Think, Philade...  united

## ❗(optional) Generate default file: filtered only Zone1 email + not company or not related to field outside medical scope

In [ ]:
#In case you want simple filtered file (filtered only Zone1 email + not company or related to field outside medical scope)

output_file_default = "filtered_email_default-17.12-all.xlsx"

# Export เป็น Excel
df_default.to_excel(output_file_default, index=False)

print(f"Exported {len(df_default)} rows to {output_file_default}")

Exported 33459 rows to filtered_email_default-17.12-all.xlsx


## ❤ **Affilitation column filtering** #เอาแค่ Email ที่มีชื่อ affiliation ตรงกับคำเฉพาะ

In [ ]:
# รับ input keyword จากผู้ใช้ (คั่นด้วย comma)
user_input = input("กรอก keyword เพื่อเลือก Affiliation (คั่นด้วย comma): ")

กรอก keyword เพื่อเลือก Affiliation (คั่นด้วย comma): surgery, surgical


In [ ]:

keywords = [k.strip() for k in user_input.split(",")]

# Create regex pattern:
# - Use re.escape to avoid issues with special characters
# - Add '.*' after the keyword to allow partial matches (e.g., "maxillo" -> "maxillofacial")
# - Use optional plural with s? but keep it flexible
pattern_list = [f"{re.escape(k)}.*" for k in keywords]
print(pattern_list)
pattern = "|".join(pattern_list)

# Filter DataFrame (case-insensitive)
df_affiliation_filtered = df_default[
    df_default['Affiliation'].str.contains(pattern, flags=re.IGNORECASE, na=False)
].reset_index(drop=True)

# Display results
print(df_affiliation_filtered.head())
print("Total rows after filtering:", len(df_affiliation_filtered))



['surgery.*', 'surgical.*']
         Name                                       EMail  \
0   Lazarides  alexanderleandros.lazarides@sinaihealth.ca   
1  Al Zadjali                         drzadjali@gmail.com   
2      Kitago                  dragonpegasus427@gmail.com   
3      Yamada                   yshinshin@shinshu-u.ac.jp   
4     Tirotta                           f.tirotta@nhs.net   

                                           Reference  \
0  How Do the Outcomes of Radiation-Associated Pe...   
1  Surgical approaches for pleomorphic adenoma of...   
2  Surgical and Oncological Outcomes of Salvage H...   
3  Efficacy of perioperative oral care management...   
4  ASO Author Reflections: Retroperitoneal Sarcom...   

                                         Affiliation Country/Region  
0  Alexander L. Lazarides (author), Division of O...         canada  
1  Fahad Al Zadjali (author), Department of ENT a...         france  
2  Minoru Kitago (author), Departments of Surgery...      

## ❗# (optional) Generate file ที่ specific keyword ใน affiliation # if you still do not want to get the file, it is no need to run

In [ ]:
# กำหนด path และชื่อไฟล์ที่ต้องการบันทึก

# Export file
output_file = input("Enter the name for the Excel file (without extension): ").strip()

# Ensure the file name ends with .xlsx
if not output_file.endswith(".xlsx"):
    output_file = output_file + ".xlsx"

df_affiliation_filtered.to_excel(output_file, index=False)
# Step 7: Download the file
files.download(output_file)

print(f"File saved as: {output_file}")
print(f"Exported {len(df_affiliation_filtered)} rows to {output_file}")


Enter the name for the Excel file (without extension): 17.12-anesthesia-cfp-susy500


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File saved as: 17.12-anesthesia-cfp-susy500.xlsx
Exported 1925 rows to 17.12-anesthesia-cfp-susy500.xlsx


In [ ]:
print(df_affiliation_filtered)

          Name                          EMail  \
0      Buckner        sb959@medschl.cam.ac.uk   
1        Smith     james.smith@menzies.edu.au   
2      Armonia       armoniaumberto@gmail.com   
3      Nutbeam      don.nutbeam@sydney.edu.au   
4         Tabb               lpp22@drexel.edu   
...        ...                            ...   
1696      West  rhonda.west@cphinnovation.org   
1697     Ojong              sojong@unicef.org   
1698  Bhandari         sbhanda8@alumni.jh.edu   
1699     Yadav        cchanchal2019@gmail.com   
1700       Kim         mek007@health.ucsd.edu   

                                              Reference  \
0     Neighbourhood planning, rural ageing and publi...   
1     Quality learning and teaching is vital for equ...   
2     Investigating Awareness Regarding Travel-Relat...   
3     Health Promotion Glossary 2021. Health Promoti...   
4     Advancing health equity in wastewater-based ep...   
...                                                 ...  

## ❤ **Reference filtering - keep only specific row that match to the input**

In [ ]:
# รับ input keyword จากผู้ใช้ (คั่นด้วย comma)
reference_input = input("กรอก keyword in Reference (คั่นด้วย comma): ")



กรอก keyword in Reference (คั่นด้วย comma): surgical oncology, cancer surgery, surgical decision making, tumor, cancer, oncology, onco, malignant


In [ ]:
keywords_ref = [k.strip() for k in reference_input.split(",")]

# Create regex pattern:
# - Use re.escape to avoid issues with special characters
# - Add '.*' after the keyword to allow partial matches (e.g., "maxillo" -> "maxillofacial")
# - Use optional plural with s? but keep it flexible
pattern_list_ref = [f"{re.escape(k)}.*" for k in keywords_ref]
print(pattern_list_ref)
pattern_ref = "|".join(pattern_list_ref)

# Filter DataFrame (case-insensitive)
df_ref_filtered = df_default[
    df_default['Reference'].str.contains(pattern_ref, flags=re.IGNORECASE, na=False)
].reset_index(drop=True)

# Display results
print(df_ref_filtered.head())
print("Total rows after filtering:", len(df_ref_filtered))



['surgical\\ oncology.*', 'cancer\\ surgery.*', 'surgical\\ decision\\ making.*', 'tumor.*', 'cancer.*', 'oncology.*', 'onco.*', 'malignant.*']
        Name                                         EMail  \
0  Lazarides    alexanderleandros.lazarides@sinaihealth.ca   
1    Murando  federica.murando@guest.policlinicogemelli.it   
2     Wilder                             c941w083@kumc.edu   
3    Magaldi                 michelangelomagaldi@gmail.com   
4       Lupe                                  klupe@toh.ca   

                                           Reference  \
0  How Do the Outcomes of Radiation-Associated Pe...   
1  Oncoplastic Breast Surgery versus Conservative...   
2  Improving Medical Student Anatomy Knowledge an...   
3  Minimal Margin Surgery and Intraoperative Neur...   
4  Evaluation of Surgical Approaches and Use of A...   

                                         Affiliation Country/Region  
0  Alexander L. Lazarides (author), Division of O...         canada  
1  Fed

## ❗# (optional) Generate file ที่ specific keyword ใน reference # if you still do not want to get the file, it is no need to run

In [ ]:
# กำหนด path และชื่อไฟล์ที่ต้องการบันทึก
output_file_ref = "filtered_ref-.xlsx"

# Export เป็น Excel
df_ref_filtered.to_excel(output_file_ref, index=False)

print(f"Exported {len(df_ref_filtered)} rows to {output_file_ref}")

Exported 417 rows to filtered_ref-.xlsx


## ⛳4) Keep only rows that shares between Filtered Affiliation Datafrom Reference Dataframe **

In [ ]:
import pandas as pd
specific_df_ref_aff=pd.merge(df_affiliation_filtered, df_ref_filtered, how='inner')

In [ ]:
print(len(df_affiliation_filtered))
print(len(df_ref_filtered))
print(specific_df_ref_aff.head())
print(len(specific_df_ref_aff))


4412
8620
        Name                                       EMail  \
0  Lazarides  alexanderleandros.lazarides@sinaihealth.ca   
1     Kitago                  dragonpegasus427@gmail.com   
2     Yamada                   yshinshin@shinshu-u.ac.jp   
3    Tirotta                           f.tirotta@nhs.net   
4       Kang                            nony87@snu.ac.kr   

                                           Reference  \
0  How Do the Outcomes of Radiation-Associated Pe...   
1  Surgical and Oncological Outcomes of Salvage H...   
2  Efficacy of perioperative oral care management...   
3  ASO Author Reflections: Retroperitoneal Sarcom...   
4  Oncologic Benefits of Neoadjuvant Treatment ve...   

                                         Affiliation Country/Region  
0  Alexander L. Lazarides (author), Division of O...         canada  
1  Minoru Kitago (author), Departments of Surgery...          japan  
2  Shin-Ichi Yamada (author), Department of Denti...          japan  
3  Fabio Tir

## 5) Generate final file to export

In [ ]:
# Export file
final_file_name = input("Enter the name for the Excel file (without extension): ").strip()

# Ensure the file name ends with .xlsx
if not final_file_name.endswith(".xlsx"):
    final_file_name = final_file_name + ".xlsx"

specific_df_ref_aff.to_excel(final_file_name, index=False)
# Step 7: Download the file
files.download(final_file_name)

print(f"File saved as: {final_file_name}")
print(f"Exported {len(specific_df_ref_aff)} rows to {final_file_name}")

Enter the name for the Excel file (without extension): 18.12-surgical-oncology-cfp


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File saved as: 18.12-surgical-oncology-cfp.xlsx
Exported 3793 rows to 18.12-surgical-oncology-cfp.xlsx


In [ ]:
# Export file - only Author emails

final_file_name2 = input("Enter the name for the Excel file (without extension): ").strip()

# Ensure the file name ends with .xlsx
if not final_file_name2.endswith(".xlsx"):
    final_file_name2 = final_file_name2 + ".xlsx"

list_email= specific_df_ref_aff["EMail"]
list_email.to_excel(final_file_name2, index=False)
# Step 7: Download the file
files.download(final_file_name2)

print(f"File saved as: {final_file_name2}")
print(f"Exported {len(list_email)} rows to {final_file_name2}")

Enter the name for the Excel file (without extension): 28.11-pediatric-ent-2-email


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File saved as: 28.11-pediatric-ent-2-email.xlsx
Exported 361 rows to 28.11-pediatric-ent-2-email.xlsx


## (Optional)# **Concatenate 2 dataframes from Reference and Affiliation (Merge 2 dataframes) that contains specific keywords of both Reference and Affiliation**

In [ ]:
import pandas as pd

merged_ref_aff = pd.concat([df_affiliation_filtered, df_ref_filtered]).drop_duplicates().reset_index(drop=True)

In [ ]:
print(merged_ref_aff.head())
print(len(merged_ref_aff ))

         Name                                       EMail  \
0   Lazarides  alexanderleandros.lazarides@sinaihealth.ca   
1  Al Zadjali                         drzadjali@gmail.com   
2      Kitago                  dragonpegasus427@gmail.com   
3      Yamada                   yshinshin@shinshu-u.ac.jp   
4     Tirotta                           f.tirotta@nhs.net   

                                           Reference  \
0  How Do the Outcomes of Radiation-Associated Pe...   
1  Surgical approaches for pleomorphic adenoma of...   
2  Surgical and Oncological Outcomes of Salvage H...   
3  Efficacy of perioperative oral care management...   
4  ASO Author Reflections: Retroperitoneal Sarcom...   

                                         Affiliation Country/Region  
0  Alexander L. Lazarides (author), Division of O...         canada  
1  Fahad Al Zadjali (author), Department of ENT a...         france  
2  Minoru Kitago (author), Departments of Surgery...          japan  
3  Shin-Ichi Yam

In [ ]:
# กำหนด path และชื่อไฟล์ที่ต้องการบันทึก
# Ask user for file name
final_file_name2 = input("Enter the name for the Excel file (without extension): ").strip()

# Ensure the file name ends with .xlsx
if not final_file_name2.endswith(".xlsx"):
    final_file_name2 = final_file_name2 + ".xlsx"

# Save DataFrame to Excel
merged_ref_aff.to_excel(final_file_name2, index=False)

print(f"File saved as: {final_file_name2}")
# Export เป็น Excel
merged_ref_aff.to_excel(final_file_name2, index=False)

print(f"Exported {len(merged_ref_aff)} rows to {final_file_name2}")

Enter the name for the Excel file (without extension): 18.12-surgical-oncology-cfp-merged-aff-ref
File saved as: 18.12-surgical-oncology-cfp-merged-aff-ref.xlsx
Exported 9239 rows to 18.12-surgical-oncology-cfp-merged-aff-ref.xlsx


## 6)  ********* From now you will get the file to export and upload in in-house system (Finder2, MRS-Editor invited) to find H-index of the scholars, IMPORT NEW FILE FROM IN-HOUSE SYSTEM AGAIN************

☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕☕

In [ ]:
from os import read
import pandas as pd
from google.colab import files

#Ask user to upload file
print("📂 Please upload your  file :")
uploaded_file = files.upload()

# Get the uploaded file name (first key in dict)
file_aff = list(uploaded_file.keys())[0]

specific_df_ref_aff = pd.read_excel(file_aff)
#print(specific_df_ref_aff.head())
print(specific_df_ref_aff.columns.tolist())
print(len(specific_df_ref_aff))

📂 Please upload your  file :


Saving 18.12-surgical-oncology-cfp.xlsx to 18.12-surgical-oncology-cfp (1).xlsx
['Name', 'EMail', 'Reference', 'Affiliation', 'Country/Region']
3793


** !!!!! In case you get H-index column from 2 sources (MRS-Author and Finder2) and will compare which source give higher H-index, then select that value to create in new H-index column**

In [ ]:
import pandas as pd

def clean_hindex(series):
    """
    Convert H-index column to numeric:
    - Non-numeric → 0
    - Blank/NaN → 0
    """
    return pd.to_numeric(series, errors='coerce').fillna(0).astype(int)

# --- Step 1: Load and process Finder2 + MRS files ---


#Ask user to upload file from Finder2
print("📂 Please upload your file from Finder2:")
uploaded_finder2 = files.upload()

# Get the uploaded file name (first key in dict)
file1 = list(uploaded_finder2.keys())[0]
df1 = pd.read_excel(file1)

#Ask user to upload file from MRS
print("📂 Please upload your file from MRS:")
uploaded_MRS = files.upload()

# Get the uploaded file name (first key in dict)
file2 = list(uploaded_MRS.keys())[0]
df2 = pd.read_excel(file2)


# Standardize column names
df1 = df1.rename(columns={"email": "Email", "h_index": "H-index"})
df2 = df2.rename(columns={"email": "Email", "h_index": "H-index"})

# Clean H-index
df1['H-index'] = clean_hindex(df1['H-index'])
df2['H-index'] = clean_hindex(df2['H-index'])

# Merge and keep max H-index
merged = pd.merge(df1[['Email', 'H-index']],
                  df2[['Email', 'H-index']],
                  on='Email', suffixes=('_Finder2', '_MRS'))
merged['H-index'] = merged[['H-index_Finder2', 'H-index_MRS']].max(axis=1)

# Final result
result = merged[['Email', 'H-index']]

# --- Step 2: Merge with specific_df_ref_aff ---
# (assumes you already loaded specific_df_ref_aff)
merged_final = specific_df_ref_aff.merge(
    result,
    left_on="EMail",
    right_on="Email",
    how="left"
).drop(columns=["Email"])

# Ensure numeric
merged_final['H-index'] = pd.to_numeric(merged_final['H-index'], errors='coerce').fillna(0).astype(int)

# Reorder columns: put H-index after Country/Region
col_order = ['Name', 'EMail', 'Reference', 'Affiliation', 'Country/Region', 'H-index']
for col in merged_final.columns:
    if col not in col_order:
        col_order.append(col)
merged_final = merged_final[col_order]

# Discount rules
def calc_discount(h):
    if h <= 1:
        return 0
    elif 2 <= h <= 4:
        return 20
    elif 5 <= h <= 14:
        return 50
    else:
        return 100

# Insert discount column right after H-index
merged_final.insert(
    merged_final.columns.get_loc("H-index") + 1,
    "discount (%)",
    merged_final["H-index"].apply(calc_discount)
)

print("✅ Done! New dataframe with H-index and discount added:")
print(merged_final.head())




📂 Please upload your file from Finder2:


#### !!!! In case you get H-index column from only on source (Finder2, MRS-Editor-invited)

In [ ]:
import pandas as pd

def clean_hindex(series):
    """
    Convert H-index column to numeric:
    - Non-numeric → 0
    - Blank/NaN → 0
    """
    return pd.to_numeric(series, errors='coerce').fillna(0).astype(int)

# --- Step 1: Upload ONE file (Finder2 or MRS) ---
print("📂 Please upload your file (Finder2 or MRS):")
uploaded_file = files.upload()

# Get the uploaded file name
file = list(uploaded_file.keys())[0]
df = pd.read_excel(file)

# Standardize column names
df = df.rename(columns={"email": "Email", "h_index": "H-index"})

# Clean H-index
df['H-index'] = clean_hindex(df['H-index'])

# Keep only Email + H-index
result = df[['Email', 'H-index']].copy()

# --- Step 2: Merge with specific_df_ref_aff ---
merged_final = specific_df_ref_aff.merge(
    result,
    left_on="EMail",
    right_on="Email",
    how="left"
).drop(columns=["Email"])

# Ensure numeric
merged_final['H-index'] = pd.to_numeric(merged_final['H-index'], errors='coerce').fillna(0).astype(int)

# Reorder columns: put H-index after Country/Region
col_order = ['Name', 'EMail', 'Reference', 'Affiliation', 'Country/Region', 'H-index']
for col in merged_final.columns:
    if col not in col_order:
        col_order.append(col)
merged_final = merged_final[col_order]

# Discount rules
def calc_discount(h):
    if h <= 1:
        return 0
    elif 2 <= h <= 4:
        return 20
    elif 5 <= h <= 14:
        return 50
    else:
        return 100

# Insert discount column right after H-index
merged_final.insert(
    merged_final.columns.get_loc("H-index") + 1,
    "discount (%)",
    merged_final["H-index"].apply(calc_discount)
)

print("✅ Done! New dataframe with H-index and discount added:")
print(merged_final.head())




In [ ]:
#print(merged_final)
merged_final = merged_final[(merged_final["discount (%)"] == 100) | (merged_final["discount (%)"] == 0)]


In [ ]:
# กำหนด path และชื่อไฟล์ที่ต้องการบันทึก
# Ask user for file name
output_path = input("Enter the name for the Excel file to save (without extension .xlsx): ").strip()

# Ensure the file name ends with .xlsx
if not output_path.endswith(".xlsx"):
    output_path = output_path + ".xlsx"

# Save DataFrame to Excel
merged_final.to_excel(output_path, index=False)

print(f"File saved as: {output_path}")
# Export เป็น Excel
merged_final.to_excel(output_path, index=False)

print(f"Exported {len(merged_final)} rows to {output_path}")
print("✅ Done! New dataframe with H-index and discount added")
print(f"📂 Output saved to: {output_path}")
print(merged_final.head())


Enter the name for the Excel file to save (without extension .xlsx): 12.12-cfp-publica-health
File saved as: 12.12-cfp-publica-health.xlsx
Exported 2421 rows to 12.12-cfp-publica-health.xlsx
✅ Done! New dataframe with H-index and discount added
📂 Output saved to: 12.12-cfp-publica-health.xlsx
        Name                          EMail  \
0    Buckner        sb959@medschl.cam.ac.uk   
1      Smith     james.smith@menzies.edu.au   
2    Armonia       armoniaumberto@gmail.com   
3    Nutbeam      don.nutbeam@sydney.edu.au   
4  Zaharatos  gerasimos.zaharatos@mcgill.ca   

                                           Reference  \
0  Neighbourhood planning, rural ageing and publi...   
1  Quality learning and teaching is vital for equ...   
2  Investigating Awareness Regarding Travel-Relat...   
3  Health Promotion Glossary 2021. Health Promoti...   
4  Monkeypox in Montréal: Epidemiology, Phylogeno...   

                                         Affiliation  Country/Region  H-index  \
0  St